<a href="https://colab.research.google.com/github/m4tt-nm/martino_DSPN_S26/blob/master/ExerciseSubmissions/10_mixed-effects-models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise 10: Mixed effects

This homework assignment is designed to give you practice fitting and interpreting mixed effects models.

We will be using the **LexicalData.csv** and **Items.csv** files from the *Homework/lexDat* folder in the class GitHub repository again.

This data is a subset of the [English Lexicon Project database](https://elexicon.wustl.edu/). It provides the reaction times (in milliseconds) of many subjects as they are presented with letter strings and asked to decide, as quickly and as accurately as possible, whether the letter string is a word or not. The **Items.csv** provides characteristics of the words used, namely frequency (how common is this word?) and length (how many letters?). Unlike in the previous homework, there isn't any missing data in the **LexicalData.csv** file.

*Data courtesy of Balota, D.A., Yap, M.J., Cortese, M.J., Hutchison, K.A., Kessler, B., Loftis, B., Neely, J.H., Nelson, D.L., Simpson, G.B., & Treiman, R. (2007). The English Lexicon Project. Behavior Research Methods, 39, 445-459.*

In [1]:
install.packages("tidyverse")
install.packages("lme4")
library(tidyverse)
library(lme4)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘rbibutils’, ‘Rdpack’, ‘minqa’, ‘nloptr’, ‘reformulas’, ‘RcppEigen’


── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.6
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.1     ✔ tibble    3.3.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.2
✔ purrr     1.2.1     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Loading required package: Matrix


Attaching package: ‘Matrix’


The following objects are masked from ‘package:tidyr’:

    expand, pack, unpack




---
## 1. Loading and formatting the data (1 point)

Load in data from the **LexicalData.csv** and **Items.csv** files. As in the previous homeworks, remove the commas from the reaction times and convert them from strings to numbers. Use `left_join` to add word characteristics `Length` and `Log_Freq_Hal` from **Items** to **LexicalData**.

*Note: the `Freq_HAL` variable in **Items.csv** has a similar formatting issue, using string values with commas. We're not going to worry about fixing this since we're only using `Log_Freq_HAL`, which is the natural log transformation of `Freq_HAL`, in this homework.*

In [7]:
# WRITE YOUR CODE HERE
lex.dat <- read.csv("https://raw.githubusercontent.com/m4tt-nm/martino_DSPN_S26/refs/heads/master/Exercise%20datasets/lexDat/LexicalData.csv")
items.dat <- read.csv("https://raw.githubusercontent.com/m4tt-nm/martino_DSPN_S26/refs/heads/master/Exercise%20datasets/lexDat/Items.csv")

lex.dat$D_RT <- as.numeric(gsub(",", "", lex.dat$D_RT))
lex.dat <- rename(lex.dat, Word = D_Word)
joined.dat <- left_join(lex.dat, items.dat, by = "Word")
head(joined.dat)


,Sub_ID,Trial,Type,D_RT,Word,Outlier,D_Zscore,Occurrences,Length,Freq_HAL,Log_Freq_HAL
,<int>,<int>,<int>,<dbl>,<chr>,<chr>,<dbl>,<int>,<int>,<chr>,<dbl>
1,157,1,1,710,browse,false,-0.437,2,6,"7,016",8.856
2,67,1,1,1094,refrigerant,false,0.825,3,11,104,4.644
3,120,1,1,587,gaining,false,-0.645,4,7,"4,039",8.304
4,21,1,1,984,cheerless,false,0.025,4,9,14,2.639
5,236,1,1,577,pattered,false,-0.763,4,8,4,1.386
6,236,2,1,715,conjures,false,-0.364,4,8,194,5.268


---
## 2. Model fitting (4 points)

First, fit a linear model with `Log_Freq_HAL` and `Length` as predictors, and `D_RT` as the output. Include an interaction term. Use `summary()` to look at the model output.

In [8]:
# WRITE YOUR CODE HERE
lex.lm <- lm(D_RT ~ Log_Freq_HAL * Length, joined.dat)
summary(lex.lm)



Call:
lm(formula = D_RT ~ Log_Freq_HAL * Length, data = joined.dat)

Residuals:
     Min       1Q   Median       3Q      Max 
-1118.01  -205.23   -86.95    90.77  3147.07 

Coefficients:
                    Estimate Std. Error t value Pr(>|t|)    
(Intercept)         610.1903    14.6678  41.601  < 2e-16 ***
Log_Freq_HAL         -6.0239     1.9678  -3.061  0.00221 ** 
Length               47.7531     1.6368  29.175  < 2e-16 ***
Log_Freq_HAL:Length  -2.9421     0.2348 -12.528  < 2e-16 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 359.1 on 62606 degrees of freedom
Multiple R-squared:  0.09473,	Adjusted R-squared:  0.09469 
F-statistic:  2184 on 3 and 62606 DF,  p-value: < 2.2e-16


Now, install `lme4` using `install.packages()` and then load the library.

In [ ]:
# WRITE YOUR CODE HERE
#I did initially, but the code would again be this:
#install.packages("lme4")
#library(lme4)


Now fit a mixed effects model that includes the same predictors as the linear model above, as well as random intercepts for `Sub_ID` (i.e., cases where subject ID shifts the RT mean). Use `summary()` to look at the model output.

In [9]:
# WRITE YOUR CODE HERE
lex.me <- lmer(D_RT ~ Log_Freq_HAL * Length + (1|Sub_ID), data = joined.dat)
summary(lex.me)


Linear mixed model fit by REML ['lmerMod']
Formula: D_RT ~ Log_Freq_HAL * Length + (1 | Sub_ID)
   Data: joined.dat

REML criterion at convergence: 888235.6

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-4.5058 -0.5472 -0.1568  0.3103 10.7381 

Random effects:
 Groups   Name        Variance Std.Dev.
 Sub_ID   (Intercept) 46333    215.3   
 Residual             82978    288.1   
Number of obs: 62610, groups:  Sub_ID, 299

Fixed effects:
                    Estimate Std. Error t value
(Intercept)         616.8445    17.1522  35.963
Log_Freq_HAL         -7.4374     1.5830  -4.698
Length               47.7477     1.3162  36.277
Log_Freq_HAL:Length  -2.8778     0.1888 -15.239

Correlation of Fixed Effects:
            (Intr) Lg_F_HAL Length
Log_Frq_HAL -0.645                
Length      -0.656  0.917         
Lg_Fr_HAL:L  0.582 -0.942   -0.923

---
## 3. Model assessment (4 points)

Compare the three t-values for the fixed effects and the mixed effects models. How do they differ, and why?

> *Each t-value in the mixed effects model is lower than the corresponding measurement in the fixed effects model. The mixed effects model takes into account between-subject variance, increasing the standard error used in the t-value calculation for each coefficient.*
>

Use the Aikeke Information Criterion (AIC) to compare these two models. Which one is better?

In [11]:
# WRITE YOUR CODE HERE
lex.ic <- AIC(lex.lm, lex.me)
lex.ic
diff(lex.ic$AIC)


,df,AIC
,<dbl>,<dbl>
lex.lm,5,914436.4
lex.me,6,888247.6


[1] -26188.82

> *The mixed effects model is better*

---
##  4. Reflection (1 point)

What other random effects could be controlled for in this data set?

> *"Word" so we could remove RT differences based on other properties of individual words (relative to other words) not measured*

**DUE:** 11:59pm EST, March 5, 2026

**IMPORTANT** Did you collaborate with anyone on this assignment? If so, list their names here.
> *No*

**GenAI Utilization** Did you utilize any generative AI tools on this assignment? If so, please list the item and the paste respective prompt you used.
>No